# 09 · Modelo final para la API y el tablero

Entrena y exporta el modelo elegido: **LightGBM con pérdida `l1`** (cuaderno 04). Para cada caso se entrenan tres modelos: el **precio de referencia** y los límites del **rango sugerido** (percentiles 7,5 y 92,5, cuaderno 07).

Hiperparámetros elegidos por validación cruzada en el cuaderno 04 (iguales para los dos conjuntos): `objective="l1"`, 1.500 árboles, `learning_rate=0.015`, `num_leaves=23`, `min_child_samples=20`, `subsample=0.8`, `colsample_bytree=0.7`, sin regularización.

| `usar_coordenadas` | Conjunto | MdAPE CV (junio) | MdAPE nuevos sept. (cuaderno 04) | Archivo |
|---|---|---|---|---|
| `False` | `v2` | 20,53 % | 20,14 % | `api/modelos/modelo_v2.joblib` |
| `True` | `v2_geo` (v2 + latitud y longitud) | 19,50 % | 17,74 % | `api/modelos/modelo_v2_geo.joblib` |

La configuración del entrenamiento está en `src/bogoprice/modelo_final.py`. Los modelos se guardan en la carpeta **`api/`**, que funciona sola: `api/predictor.py` recibe los datos crudos de Inside Airbnb, hace toda la transformación de variables y predice, sin depender del paquete `bogoprice`. Las corridas van al experimento `bogoprice-modelo-final` de MLflow.

In [1]:
import pandas as pd

from bogoprice import modelo_final
from bogoprice.experimentos import preparar_datos
from bogoprice.tracking import iniciar_mlflow, registrar_corrida

# Tag de autor de MLflow
AUTOR = "Edgar"
iniciar_mlflow("bogoprice-modelo-final")

## 1. Validación

Antes del entrenamiento final se verifica la configuración con el mismo esquema de los demás cuadernos: entrenar con junio y evaluar con septiembre. Estas métricas quedan guardadas dentro del paquete del modelo, para mostrarlas en el panel de monitoreo.

In [2]:
datos = preparar_datos()
validacion = {}
for usar_coordenadas in [False, True]:
    version = modelo_final.version_de(usar_coordenadas)
    m = modelo_final.validar(usar_coordenadas, datos)
    validacion[version] = m
    registrar_corrida(
        f"validacion_{version}",
        {"modelo": "LightGBM", "conjunto": version, "entrenamiento": "2026-06",
         "evaluacion": "2026-09", "percentiles": str(modelo_final.PERCENTILES),
         **modelo_final.PARAMETROS_PRECIO[version]},
        m, tags={"autor": AUTOR, "conjunto": version, "etapa": "validacion"})

pd.DataFrame(validacion).T.round(2)

P99 junio: $1,330,087 | train 17749 | test 17561 | nuevos 1632 | localidades 9 | tipologías 17


🏃 View run validacion_v2 at: http://52.91.31.34:5000/#/experiments/1/runs/3bbd8cf14c6546fea62f4403c11210e7
🧪 View experiment at: http://52.91.31.34:5000/#/experiments/1


🏃 View run validacion_v2_geo at: http://52.91.31.34:5000/#/experiments/1/runs/99d5d60427e845819c73e1cc649d0ce1
🧪 View experiment at: http://52.91.31.34:5000/#/experiments/1


,mdape,mdape_nuevos,cobertura,cobertura_nuevos,ancho_mediano_cop
v2,18.31,20.14,81.98,81.50,139982.59
v2_geo,16.45,17.74,81.19,80.51,120268.12


## 2. Entrenamiento final

Se entrena con **junio y septiembre juntos**, como se definió en el alcance. Si un anuncio aparece en las dos capturas, se usa su observación más reciente. El límite de atípicos, las localidades habilitadas y las tipologías se recalculan con los datos unidos.

In [3]:
df_final, info = modelo_final.preparar_entrenamiento_final()
print(f"Anuncios: {info['n_anuncios']} | P99: ${info['limite_p99']:,.0f} | "
      f"localidades: {len(info['localidades'])} | tipologías: {len(info['tipologias'])}")

rutas = {}
for usar_coordenadas in [False, True]:
    version = modelo_final.version_de(usar_coordenadas)
    paquete = modelo_final.entrenar_paquete(usar_coordenadas, df_final, info, validacion[version])
    ruta = modelo_final.guardar_paquete(paquete)
    rutas[version] = ruta
    registrar_corrida(
        f"final_{version}",
        {"modelo": "LightGBM", "conjunto": version, "capturas": "+".join(info["capturas"]),
         "n_anuncios": info["n_anuncios"], **modelo_final.PARAMETROS_PRECIO[version]},
        {f"validacion_{k}": v for k, v in validacion[version].items()},
        pipeline=paquete["modelo"], artefactos=[str(ruta)],
        tags={"autor": AUTOR, "conjunto": version, "etapa": "final"})
    print(f"{version:7s} guardado en {ruta} ({ruta.stat().st_size / 1e6:.1f} MB)")

# Versiones exactas de las librerías para la API
print("Requisitos de la API:", modelo_final.escribir_requirements_api(paquete))

Anuncios: 18932 | P99: $1,308,339 | localidades: 9 | tipologías: 17


2026/11/24 05:04:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/24 05:04:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_v2 at: http://52.91.31.34:5000/#/experiments/1/runs/7c3acf8462dd47369593b283704e34be
🧪 View experiment at: http://52.91.31.34:5000/#/experiments/1
v2      guardado en /home/ubuntu/precios_airbnb_bogota/api/modelos/modelo_v2.joblib (2.2 MB)


2026/11/24 05:04:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/24 05:04:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run final_v2_geo at: http://52.91.31.34:5000/#/experiments/1/runs/fc712bfb61774404b5cf68374e6124fa
🧪 View experiment at: http://52.91.31.34:5000/#/experiments/1
v2_geo  guardado en /home/ubuntu/precios_airbnb_bogota/api/modelos/modelo_v2_geo.joblib (2.4 MB)
Requisitos de la API: /home/ubuntu/precios_airbnb_bogota/api/requirements.txt


## 3. Uso de la carpeta `api/` con datos crudos

`api/predictor.py` se importa directamente desde la carpeta `api/`, sin el paquete `bogoprice`. Recibe filas del CSV crudo de Inside Airbnb, tal como vienen, y devuelve el precio de referencia y el rango sugerido.

In [4]:
import sys
sys.path.insert(0, str(modelo_final.CARPETA_API))
from predictor import Predictor

from bogoprice.data_loader import cargar_y_limpiar

predictor = Predictor()
crudo = cargar_y_limpiar("2026-09")          # CSV crudo de septiembre

columnas_crudas = ["id", "neighbourhood_cleansed", "room_type", "property_type", "accommodates",
                   "bedrooms", "beds", "bathrooms_text", "amenities", "minimum_nights",
                   "latitude", "longitude"]
muestra = crudo[crudo["neighbourhood_cleansed"].isin(["Chapinero", "Usaquen", "Teusaquillo"])]
muestra = muestra.sample(5, random_state=1)

pred = predictor.predecir(muestra[columnas_crudas], usar_coordenadas=True)
pd.concat([muestra[["id", "neighbourhood_cleansed", "room_type", "accommodates", "price"]], pred],
          axis=1)

,id,neighbourhood_cleansed,room_type,accommodates,price,precio_referencia,rango_inferior,rango_superior,motivo_no_estimado,modelo
18503,1731227782935898702,Chapinero,Entire home/apt,2,168000.00,151400.0,112000.0,234500.0,,LightGBM v2_geo
2723,41835719,Teusaquillo,Entire home/apt,2,92071.50,86900.0,62800.0,145400.0,,LightGBM v2_geo
1507,28342311,Chapinero,Entire home/apt,3,390222.61,288000.0,198900.0,443600.0,,LightGBM v2_geo
11784,1338721136732251579,Teusaquillo,Entire home/apt,2,141887.00,103700.0,73100.0,149300.0,,LightGBM v2_geo
18659,1739458941044803535,Usaquen,Entire home/apt,3,187193.75,127200.0,88100.0,191000.0,,LightGBM v2_geo


In [5]:
# Un anuncio como diccionario (lo que enviaría el formulario del tablero)
anuncio = {
    "neighbourhood_cleansed": "Chapinero",
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "accommodates": 2,
    "bedrooms": 1,
    "beds": 1,
    "bathrooms_text": "1 bath",
    "amenities": ["Wifi", "Kitchen", "Elevator", "Dedicated workspace", "Hot water", "TV"],
    "minimum_nights": 2,
    "latitude": 4.6486,     # solo se usan con usar_coordenadas=True
    "longitude": -74.0603,
}

pd.DataFrame({
    "sin coordenadas": predictor.predecir(anuncio, usar_coordenadas=False),
    "con coordenadas": predictor.predecir(anuncio, usar_coordenadas=True),
})

,sin coordenadas,con coordenadas
precio_referencia,164100.0,149300.0
rango_inferior,116700.0,130200.0
rango_superior,258700.0,220900.0
moneda,COP,COP
modelo,LightGBM v2,LightGBM v2_geo
usar_coordenadas,False,True
mdape_validacion,20.1,17.7
cobertura_rango_validacion,81.5,80.5


In [6]:
# Aporte de cada variable al precio (gráfico de contribución del tablero)
explicacion = predictor.explicar(anuncio, usar_coordenadas=True)
print(f"Precio base: ${explicacion.attrs['precio_base_cop']:,.0f}  ->  "
      f"precio de referencia: ${explicacion.attrs['precio_referencia_cop']:,.0f}")
explicacion.head(10)

Precio base: $156,359  ->  precio de referencia: $149,297


,variable,efecto_pct,efecto_cop
0,Tipo de alojamiento,9.3,12700.0
1,Baños,-6.5,-10400.0
2,Huéspedes,-5.7,-9000.0
3,Localidad,4.5,6500.0
4,Noches mínimas,4.3,6100.0
5,Habitaciones,-3.5,-5400.0
6,Ubicación exacta,-3.1,-4800.0
7,Parqueadero,-1.9,-2800.0
8,Ascensor,1.6,2400.0
9,Balcón o patio,-1.1,-1700.0


## 4. Verificación: la API transforma igual que el entrenamiento

Se toman los anuncios de septiembre, se transforman por los dos caminos (el del entrenamiento, con `features.py` y `variables_v2.py`, y el de `api/predictor.py`) y se comparan las predicciones. Deben ser idénticas.

In [7]:
import numpy as np
from bogoprice import variables_v2
from bogoprice.features import construir_features

crudo_validos = crudo[crudo["id"].isin(datos.test["id"])]
for usar_coordenadas in [False, True]:
    p = predictor.paquete(usar_coordenadas)
    entrenamiento = variables_v2.agregar_variables_v2(construir_features(crudo_validos),
                                                      p["tipologias"])[p["columnas"]]
    X_api, motivo = predictor.preparar(crudo_validos, usar_coordenadas)
    ok = motivo.eq("")
    diferencia = np.abs(p["modelo"].predict(entrenamiento[ok]) - p["modelo"].predict(X_api[ok])).max()
    print(f"{p['version']:7s} anuncios: {ok.sum()} | diferencia máxima en predicción: {diferencia:.2e}")
    assert diferencia < 1e-9, "La transformación de la API no coincide con la del entrenamiento"
print("La transformación de api/predictor.py es idéntica a la del entrenamiento.")

v2      anuncios: 17561 | diferencia máxima en predicción: 0.00e+00


v2_geo  anuncios: 17561 | diferencia máxima en predicción: 0.00e+00
La transformación de api/predictor.py es idéntica a la del entrenamiento.
